In [1]:
!pip install -q --upgrade openai python-dotenv gradio

In [2]:
import os
import warnings
from IPython.display import display, Markdown, HTML  # For displaying HTML directly
from dotenv import load_dotenv

from openai import OpenAI

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

# OpenAI Client (Refresher)
openai_client = OpenAI(api_key = openai_api_key)
print(f"OpenAI Client configured (Key starts with: {openai_api_key[:5]}...).")


OpenAI Client configured (Key starts with: sk-pr...).


In [3]:
# Helper function to display markdown nicely ---
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))


In [4]:
def display_html_code(provider_name, html_content):
    """Displays generated HTML code block nicely."""
    print_markdown(f"### Generated HTML from {provider_name}:")
    # Display as a formatted code block
    display(Markdown(f"```html\n{html_content}\n```"))

In [5]:
test_prompt = "Build a function in Python that calculates BMI. Let users choose between metric (kg/m) and imperial (lb/in) units."
response = openai_client.chat.completions.create(model = "gpt-4o", 
                                                 messages = [{"role": "user", 
                                                              "content": test_prompt}],
                                                 temperature = 0.5)
print_markdown(response.choices[0].message.content)

To create a function in Python that calculates BMI and allows users to choose between metric and imperial units, you can follow the code structure below:

```python
def calculate_bmi(weight, height, unit='metric'):
    """
    Calculate the Body Mass Index (BMI).
    
    :param weight: Weight of the person. In kilograms if metric, pounds if imperial.
    :param height: Height of the person. In meters if metric, inches if imperial.
    :param unit: The unit system to use ('metric' or 'imperial').
    :return: The calculated BMI.
    """
    if unit == 'metric':
        # BMI = weight in kg / (height in m)^2
        bmi = weight / (height ** 2)
    elif unit == 'imperial':
        # BMI = (weight in lb / (height in inches)^2) * 703
        bmi = (weight / (height ** 2)) * 703
    else:
        raise ValueError("Invalid unit system. Use 'metric' or 'imperial'.")
    
    return bmi

# Example usage:
weight_metric = 70  # in kilograms
height_metric = 1.75  # in meters
bmi_metric = calculate_bmi(weight_metric, height_metric, unit='metric')
print(f"Metric BMI: {bmi_metric:.2f}")

weight_imperial = 154  # in pounds
height_imperial = 69   # in inches
bmi_imperial = calculate_bmi(weight_imperial, height_imperial, unit='imperial')
print(f"Imperial BMI: {bmi_imperial:.2f}")
```

### Explanation:
- The function `calculate_bmi` takes three parameters: `weight`, `height`, and `unit`.
- The `unit` parameter defaults to `'metric'`, which means the function will calculate BMI using the metric system unless specified otherwise.
- The metric formula for BMI is \( \text{BMI} = \frac{\text{weight in kg}}{(\text{height in m})^2} \).
- The imperial formula for BMI is \( \text{BMI} = \left(\frac{\text{weight in lb}}{(\text{height in inches})^2}\right) \times 703 \).
- The function raises a `ValueError` if an invalid unit system is provided.
- Example usage is provided to demonstrate how to calculate BMI using both metric and imperial units.

##  DEFINING THE STARTUP IDEA & PROMPT

We need a consistent prompt to give each AI model. This prompt should clearly state:
1.  The **context** or personality we want the AI to take (e.g., You are an expert web developer).
2.  The **instruction**: generate HTML code for a landing page.
3.  The **output indicator** required: specifically, the full HTML structure for an `index.html` file.

Let's define our startup idea and the prompt.

In [6]:
# Define the startup name and concept
startup_name = "Quantum Leap Computing"
startup_concept = "A cloud platform providing affordable access to quantum computing resources for researchers and small businesses. Focus on a user-friendly interface and educational materials."

In [7]:
# Define the core prompt for the LLMs
# We explicitly ask for HTML code and specify the file name 'index.html'
html_prompt = f"""
You are a helpful AI assistant acting as a front-end web developer.

Your task is to generate the complete HTML code for a simple, clean, and professional-looking landing page (index.html) for a new startup.

Startup Name: {startup_name}
Concept: {startup_concept}

Please generate ONLY the full HTML code, starting with <!DOCTYPE html> and ending with </html>.
Create a modern, visually appealing landing page with the following:

1. A sleek header with the startup name in a bold, modern font and a compelling tagline
2. A hero section with a clear value proposition and call-to-action button
3. A features section highlighting 3-4 key benefits with icons or simple visuals
4. A "How it Works" section with numbered steps
5. A testimonials section with fictional customer quotes
6. A pricing section with at least two tiers
7. A professional footer with navigation links and social media icons

Use inline CSS for styling with a modern color palette (primary, secondary, and accent colors). 
Include responsive design elements, subtle animations, and whitespace for readability.
Emphasize AI capabilities, ease of use, and business benefits throughout the copy.
Focus on conversion-optimized marketing messages that highlight pain points and solutions.

Do not include any explanations before or after the code block. Just provide the raw HTML code.
"""

print_markdown("**Core Prompt defined for the LLMs:**")
print_markdown(f"> {html_prompt}")  # Print the start of the prompt to verify

**Core Prompt defined for the LLMs:**

> 
You are a helpful AI assistant acting as a front-end web developer.

Your task is to generate the complete HTML code for a simple, clean, and professional-looking landing page (index.html) for a new startup.

Startup Name: Quantum Leap Computing
Concept: A cloud platform providing affordable access to quantum computing resources for researchers and small businesses. Focus on a user-friendly interface and educational materials.

Please generate ONLY the full HTML code, starting with <!DOCTYPE html> and ending with </html>.
Create a modern, visually appealing landing page with the following:

1. A sleek header with the startup name in a bold, modern font and a compelling tagline
2. A hero section with a clear value proposition and call-to-action button
3. A features section highlighting 3-4 key benefits with icons or simple visuals
4. A "How it Works" section with numbered steps
5. A testimonials section with fictional customer quotes
6. A pricing section with at least two tiers
7. A professional footer with navigation links and social media icons

Use inline CSS for styling with a modern color palette (primary, secondary, and accent colors). 
Include responsive design elements, subtle animations, and whitespace for readability.
Emphasize AI capabilities, ease of use, and business benefits throughout the copy.
Focus on conversion-optimized marketing messages that highlight pain points and solutions.

Do not include any explanations before or after the code block. Just provide the raw HTML code.


## GENERATE HTML LANDING PAGES WITH OPENAI

In [8]:
# Generate HTML using OpenAI gpt-4o model
openai_html_output = "<!-- OpenAI generation not run or failed -->" 
print_markdown("## Calling OpenAI ...")
try:
    response = openai_client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            # No system prompt needed here as instructions are in the user prompt
            {"role": "user", "content": html_prompt}
        ],
        temperature = 0.5,  # A bit deterministic for code generation
    )
    openai_html_output = response.choices[0].message.content

    # Sometimes OpenAI might wrap the code in markdown fences, remove that if present
    if openai_html_output.strip().startswith("```html"):
        lines = openai_html_output.strip().splitlines()
        openai_html_output = "\n".join(lines[1:-1]).strip()
    else:
        openai_html_output = openai_html_output.strip()

    # Display the generated HTML code
    display_html_code("OpenAI (gpt-4o)", openai_html_output)

    # Save the output to a file
    file_path = "openai_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(openai_html_output)
    print_markdown(f"Successfully saved OpenAI output to `{file_path}`")


except Exception as e:
    print_markdown(f"Error calling OpenAI API: {e}")
    openai_html_output = f"<!-- Error calling OpenAI API: {e} -->"

## Calling OpenAI ...

### Generated HTML from OpenAI (gpt-4o):

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Quantum Leap Computing</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 0;
            color: #333;
            line-height: 1.6;
        }
        header {
            background-color: #2C3E50;
            color: #ECF0F1;
            padding: 20px;
            text-align: center;
        }
        header h1 {
            margin: 0;
            font-size: 2.5em;
        }
        header p {
            font-size: 1.2em;
            margin-top: 10px;
        }
        .hero {
            background-color: #3498DB;
            color: #fff;
            padding: 60px 20px;
            text-align: center;
        }
        .hero h2 {
            font-size: 2em;
            margin-bottom: 20px;
        }
        .hero button {
            background-color: #E74C3C;
            color: #fff;
            border: none;
            padding: 15px 30px;
            font-size: 1em;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .hero button:hover {
            background-color: #C0392B;
        }
        .features, .how-it-works, .testimonials, .pricing {
            padding: 40px 20px;
            text-align: center;
        }
        .features h3, .how-it-works h3, .testimonials h3, .pricing h3 {
            font-size: 1.8em;
            margin-bottom: 20px;
        }
        .features div, .how-it-works div {
            display: inline-block;
            width: 30%;
            margin: 1.5%;
            vertical-align: top;
        }
        .features div img, .how-it-works div img {
            width: 50px;
            height: 50px;
            margin-bottom: 10px;
        }
        .testimonials blockquote {
            font-style: italic;
            margin: 20px auto;
            max-width: 600px;
        }
        .pricing table {
            width: 100%;
            max-width: 600px;
            margin: 0 auto;
            border-collapse: collapse;
        }
        .pricing th, .pricing td {
            border: 1px solid #ddd;
            padding: 10px;
        }
        .pricing th {
            background-color: #f4f4f4;
        }
        footer {
            background-color: #2C3E50;
            color: #ECF0F1;
            padding: 20px;
            text-align: center;
        }
        footer a {
            color: #ECF0F1;
            margin: 0 10px;
            text-decoration: none;
        }
        footer .social-icons img {
            width: 24px;
            height: 24px;
            margin: 0 5px;
        }
        @media (max-width: 768px) {
            .features div, .how-it-works div {
                width: 80%;
                margin: 10% auto;
            }
        }
    </style>
</head>
<body>

<header>
    <h1>Quantum Leap Computing</h1>
    <p>Unlock the Power of Quantum Computing for Everyone</p>
</header>

<section class="hero">
    <h2>Affordable Quantum Computing for Researchers and Small Businesses</h2>
    <button>Get Started Now</button>
</section>

<section class="features">
    <h3>Features</h3>
    <div>
        <img src="icon1.png" alt="Feature 1">
        <h4>Easy to Use</h4>
        <p>Our platform is designed with user-friendliness in mind, allowing you to focus on your research.</p>
    </div>
    <div>
        <img src="icon2.png" alt="Feature 2">
        <h4>Cost-Effective</h4>
        <p>Access powerful quantum computing resources at a fraction of the cost.</p>
    </div>
    <div>
        <img src="icon3.png" alt="Feature 3">
        <h4>Educational Resources</h4>
        <p>Learn and grow with our comprehensive educational materials.</p>
    </div>
</section>

<section class="how-it-works">
    <h3>How it Works</h3>
    <div>
        <img src="step1.png" alt="Step 1">
        <h4>Step 1</h4>
        <p>Create an account and choose your plan.</p>
    </div>
    <div>
        <img src="step2.png" alt="Step 2">
        <h4>Step 2</h4>
        <p>Access our cloud-based quantum computing platform.</p>
    </div>
    <div>
        <img src="step3.png" alt="Step 3">
        <h4>Step 3</h4>
        <p>Start computing and get results quickly.</p>
    </div>
</section>

<section class="testimonials">
    <h3>Testimonials</h3>
    <blockquote>
        "Quantum Leap Computing has revolutionized our research capabilities. The platform is intuitive and the pricing is unbeatable."
        <br><strong>- Dr. Alice Quantum</strong>
    </blockquote>
    <blockquote>
        "As a small business owner, I never thought I could afford quantum computing. Quantum Leap made it possible!"
        <br><strong>- Bob Smith, Tech Startup CEO</strong>
    </blockquote>
</section>

<section class="pricing">
    <h3>Pricing</h3>
    <table>
        <thead>
            <tr>
                <th>Plan</th>
                <th>Price</th>
                <th>Features</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Basic</td>
                <td>$29/month</td>
                <td>Access to basic quantum computing resources, community support</td>
            </tr>
            <tr>
                <td>Pro</td>
                <td>$99/month</td>
                <td>All features of Basic, plus premium support and advanced tools</td>
            </tr>
        </tbody>
    </table>
</section>

<footer>
    <p>&copy; 2023 Quantum Leap Computing. All rights reserved.</p>
    <a href="#">Home</a> |
    <a href="#">Features</a> |
    <a href="#">Pricing</a> |
    <a href="#">Contact</a>
    <div class="social-icons">
        <a href="#"><img src="facebook.png" alt="Facebook"></a>
        <a href="#"><img src="twitter.png" alt="Twitter"></a>
        <a href="#"><img src="linkedin.png" alt="LinkedIn"></a>
    </div>
</footer>

</body>
</html>
```

Successfully saved OpenAI output to `openai_landing_page.html`